In [ ]:
import os
import requests, base64
import time

In [ ]:
#---------  A decorator to calculate the inference time-----------------
def time_it(func):
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        print(f"{func.__name__} executed in {time.time() - start:.2f}s")
        return result
    return wrapper

In [ ]:
from google.colab import userdata
os.environ["NVIDIA_API_KEY"] = "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"

In [ ]:
# variables specific to your CoLabsetup ----------------------------------------
from google.colab import drive
import os, sys
drive.mount('/content/drive')
dir = "/content/drive/My Drive/test_images/images_05_14" # Here goes the folder path to the images files
image_files = os.listdir(dir)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
def create_strict_satellite_prompt(filename: str, questions: list[str]) -> str:
    import os

    basename = os.path.splitext(filename)[0]
    parts = basename.split("_")

    # Check for band index image
    if len(parts) == 3 and parts[1].isalpha() and '-' in parts[2]:
        location, operation, date = parts
        image_type = f"{operation.upper()} result from the bands of a Sentinel-2 image"
    # Check for RGB image
    elif len(parts) == 2 and '-' in parts[1]:
        location, date = parts
        image_type = "Sentinel-2 RGB composite image"
    else:
        return "Error: Filename format not recognized."

    questions_block = "\n".join(f"- {q}" for q in questions)

    return f"""
You are analyzing a Sentinel-2 satellite image.

Filename: {filename}
Location: {location}
Date: {date}
Image Type: {image_type}

Strict analysis rules:
- Base your interpretation solely on observable environmental conditions in the image.
- Use only the location name from the filename ({location}). Do not refer to or infer any other geography.
- Do not provide generalized or assumed information not directly observable.
- If the image shows results from band operations (e.g., NDVI, FMI), interpret the specific indicators accordingly.

Answer the following, strictly using the image and metadata above:
{questions_block}
""".strip()


In [ ]:
def LlamaPromptGenerator(filename: str, question: str) -> str:
    info = os.path.splitext(filename)[0]
    basename = info.split('/')[-1]
    parts = basename.split("_")

    # Check for band index image
    if len(parts) == 3 and parts[1].isalpha() and '-' in parts[2]:
        location, operation, date = parts
        image_type = f"{operation.upper()} result from the bands of a Sentinel-2 image"
    # Check for RGB image
    elif len(parts) == 2 and '-' in parts[1]:
        location, date = parts
        image_type = "Sentinel-2 RGB composite image"
    else:
        return "Error: Filename format not recognized."


    prompt =  f"""
You are analyzing a Sentinel-2 satellite image.

Filename: {basename}
Location: {location}
Date: {date}
Image Type: {image_type}

Strict analysis rules:
- Base your interpretation solely on observable environmental conditions in the image.
- Use only the location name from the filename ({location}). Do not refer to or describe any other geographic location.
- Do not provide generalized or assumed information not directly observable in the image.
- If the image is a result of band operations (e.g., NDVI, FMI), interpret the specific indicators accordingly.

Answer the following, strictly using the image and metadata above:
{question}


""".strip()

    if multi_shot_examples:
        return f"{multi_shot_examples.strip()}\n\n{prompt}"
    else:
        return prompt

In [ ]:
multi_shot_examples = """
Example 1:
Filename: Amsterdam_rgb_2024-02-01.png
Location: Amsterdam
Date: 2024-02-01
Image Type: RGB, color image depicting the city of Amsterdam.
Q: What does an RGB image illustrate?
A: An RGB image is a composite of the red, green, and blue bands of a satellite image. It shows the scene as it might appear to the human eye from low Earth orbit.
Here, the RGB image shows the cityscape of Amsterdam on a cloudless day in the winter.

Example 2:
Filename: Paris_ndvi_2025-02-01.png
Location: Paris
Date: 2025-02-01
Image Type: NDVI, normalized difference vegetation index, calculated from the bands of a Sentinel-2 image.
Q: What does an NDVI image describe?
A: The NDVI image shows the extent of vegetation across a landscape based on the presence of chlorophyll in plant leaves. NDVI utilized the difference between
the rear infrared (NIR) and red (RED) bands. NDVI values range from -1 to 1, with positive values indicating high chlorophyl concentration and negative values
indicating low chlorophyll concentration.
Here, the image shows the extent of vegetation coverage across Paris. As this image was collected in the winter of the northern hemisphere, the meager vegetation evident
 is due to winter dormancy of plant life.

Example 3:
Filename: Islamabad_fmi_2024-01-13.png
Location: Islamabad
Date: 2024-01-13
Image Type: FMI, ferrous metal index, calculated from the bands of a Sentinel-2 image.
Q: What does the FMI image show?
A: The FMI image illustrates the concentration of iron-bearing minerals, specifically iron oxide. It does this by utilizing the reflectance differences
between shortwave infrared (SWIR) and near infrared (NIR) bands. FMI values range from -1 to 1, with positive values indicating high iron concentration and
negative values indicating low iron concentration.
Here, the image shows a high concentration of ferrous minerals in a lake in the vicinity of Islamabad. Iron ore is an essential raw material for the
production of steel. It is also used in the manufacture of fertilizers and other agricultural products. The presence of this heavy metal in the area
suggests that it could be a result of mining, processing, or transportation of the ore.

Example 4:
Filename: Bagdad_nbdi_2024-06-01.png
Location: Bagdad
Date: 2024-06-01
Image Type: NDBI, normalized built-up index, calculated from the bands of a Sentinel-2 image.
Q: What does the NDBI image show?
A: NDBI is used to identify map built-up areas from satellite imagery. It works by comparing the reflectance values in the near infrared (NIR)
and shortwave infrared (SWIR) bands, where built-up areas tend to have higher SWIR and lower NIR reflectance. NDBI values range from -1 to 1, with positive
values indicating densely built-up areas and negative values indicating non-built-up areas like vegetation, water, or bare soil.
Here, the image shows high built-up density across the center of the city, suggesting intense urbanization.

Example 5:
Filename: LosAngeles_nbi_2025-01-12.png
Location: LosAngeles
Date: 2025-01-12
Image Type: NBI, normalized burn index, calculated from the bands of a Sentinel-2 image.
Q: What does the NBI image show?
A: The Normalized Burn Index, or Normalized Burn Ratio (NBI or NBR) is an index used in Sentinel-2 images for identifying and assessing
areas affected by fire and can indicate burn severity. NBI uses the near infrared (NIR) and shortwave infrared (SWIR) bands. NBI values range between
 -1 and 1. Healthy vegetation exhibits high NBI values, while burned areas show low values.
Here, the image shows burnt areas in Los Angeles after the January 2025 Southern California wildfires.
""".strip()


In [ ]:
"""
  	Bullet Points vs. Full Sentences:
    Bullet Points: When using bullet points, you're breaking down information into distinct, digestible pieces.
    This can help the model focus on individual aspects of your prompt, leading to more granular responses.
    However, bullet points may sometimes result in outputs that are disjointed or lack coherence if the model fails to connect the dots.

    Full Sentences: Structuring your prompt in full sentences often leads to more fluid and connected responses.
    The model is better able to understand context and relationships between different pieces of information.
    This method is particularly effective when seeking more nuanced or detailed outputs.

"""
#[Source: https://www.reddit.com/r/PromptEngineering/comments/1exltu2/prompt_structure_and_bullet_points/] Not the oficial source but recommended by the developers community


sys_prompt_7b = """
You are an expert assistant specializing in the interpretation of Sentinel-2 satellite imagery for environmental analysis.

You analyze spectral data from RGB and Near-Infrared bands and understand the environmental significance of band operations such as:
- NDVI (vegetation health)
- NDWI (water content)
- BAI (burn severity)
- Clay Minerals Index
- Ferrous Minerals Index (FMI)

Only describe observable environmental conditions based on the spectral content of the supplied image.

FILENAME RULES:
- The first word in the image filename is the location. Only reference this location — do not mention any other places.
- If the filename includes a code like NDVI, NDWI, BAI, FMI, or others after the location name, it indicates a band-derived index.
- If no index is specified, the image is an RGB composite.

EXAMPLES:
- Islamabad_2025_01_24.png → RGB image of Islamabad
- Paris_NDVI_2025-02-01.png → NDVI image of Paris
- Islamabad_FMI_2025-01-24.png → FMI image of Islamabad

INSTRUCTIONS:
- Do not include general statements about cities, landscapes, or regions.
- Do not speculate or provide assumed information.
- Your response must be based **only** on what is visually inferable from the image type and spectral content.
- Limit your response to **100 words**. Insert a newline after approximately 30 words.
- Do not use bullet points or lists. Write in complete sentences using concise and formal prose.

Your knowledge cutoff is August 2024.
"""


In [ ]:
sys_prompt_7a = """You are an expert-level assistant specializing in the analysis of geospatial satellite imagery from the European Union's Sentinel-2 satellites.
You also are an expert-level assistant in Geography and can recognize city names and associate them with the satellite images.

Your focus is the identification of possible signs of environmental degradation in satellite imagery. To that end, you interpret satellite images spectra including
Red, Green, Blue, NearInfrared bands. You understand the significance of band operations such as Normalized Difference Vegetation Index (NDVI),
Normalized Difference Water Index (NDWI), Burn Area Index (BAI), Clay Minerals Index and Ferrous Minerals Index (FMI). You understand how these indeces can be used to evaluate environmental
conditions.

Describe only the environmental conditions observable in the Sentinel-2 satellite images supplied to you.
The first word of the image title designates the location. Reference only that location in our analysis.
EXAMPLE:
Islamabad_2025_01_24.png is a Sentinel_2 image of Islamabad, Pakistan.

IF the image is the result of a band operation (such as NDVI) then that code is included in the title after the location name. Otherwise, the image is an RGB composite.
EXAMPLES:
Paris_NDVI_2025-02-01.png is a NDVI result from the bands of a Sentinel_2 image of Paris, France.
Islamabad_FMI_2025-01-24.png is an FMI result from the bands of a Sentinel_2 image of Islamabad, Pakistan.

DO NOT include generic observations about landscapes or cities. Formulate your responses as complete sentences in well-constructed prose of maximumly 100 words.
Add a newline after 30 words. DO NOT use bullet points or numbered lists in your description.

Your knowledge base has a cuttoff of August 2024."""

In [ ]:
# KOSMOS system prompt
sys_prompt_10 = """
You are an expert assistant specializing in the interpretation of Sentinel-2 satellite imagery for environmental analysis.

You analyze spectral data from RGB and Near-Infrared bands and understand the environmental significance of band operations such as:
- NDVI (vegetation health)
- NDWI (water content)
- NDBI (built-up area)
- NBR  (burn ratio)
- FMI  (ferrous minerals)

Only describe observable environmental conditions based on the spectral content of the supplied image.

FILENAME RULES:
- The first word in the image filename is the location. Only reference this location — do not mention any other places.
- The second word indicates the image type: rgb, ndvi, ndwi, nbr, fmi, or others after the location name, it indicates a band-derived index.
- If no index is specified, the image is an RGB composite.

EXAMPLES:
- Islamabad_rgb_2024_01_24.png → RGB image of Islamabad from Janary 24th, 2024.
- Paris_ndvi_2025-02-01.png → NDVI image of Paris from Febuary 1st, 2025.
- Buenos Aires_nbr_2024-12-28.png → NBR image of Buenos Aires from December 28th, 2024.
- Islamabad_fmi_2024-01-24.png → FMI image of Islamabad from Janary 24th, 2024.

INSTRUCTIONS:
- Do not include general statements about cities, landscapes, or regions.
- Do not speculate or provide assumed information.
- Your response must be based **only** on what is visually inferable from the image type and spectral content.
- Limit your response to **100 words**. Insert a newline after approximately 30 words.
- Do not use bullet points or lists. Write in complete sentences using concise and formal prose.

Your knowledge cutoff is August 2024.
"""

In [ ]:
model_name = 'meta/llama-4-maverick-17b-128e-instruct'

SYSTEM_PROMPT = sys_prompt_10

In [ ]:

questions = ["Which environmental hazards are present in this Sentinel-2 satellite image?"]
#questions = ["What can you detect in this Sentinel-2 satellite image?",
  #         "Which environmental hazards are present in this Sentinel-2 satellite image?"]

In [ ]:
@time_it
def generateCaptions_llama4(image_file_path, prompt):
    invoke_url = "https://integrate.api.nvidia.com/v1/chat/completions"
    stream = False # Put stream to False, dont want the model to be a conversational agent for now
    #--------Opens image, base64 encodes it, converting the raw bytes into a text string---------------------
    #--------image_b64 contains the base64 string version of your image----------------
    try:
        with open(image_file_path, "rb") as f:
            image_b64 = base64.b64encode(f.read()).decode()
    except Exception as e:
        print(f"Error processing {image_file_path}: {e}")

    #--------Defining API key------------------------------------------
    #--------API key is given followed by the string "Bearer"

    headers = {
    "Authorization": f"Bearer {os.getenv('NVIDIA_API_KEY')}",
     "Accept": "text/event-stream" if stream else "application/json"
    }

    #--------Model initialization with system prompt --------------------------------------
    # Info on sentinel-2 band operations
    # https://pro.arcgis.com/en/pro-app/3.3/help/analysis/raster-functions/band-arithmetic-function.htm
    # https://www.sciencedirect.com/science/article/pii/S0303243421000507

    payload = {
    "model": model_name,
    "messages":
        [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f""" {prompt} <img src="data:image/png;base64,{image_b64}" />"""}
        ],
    "max_tokens": 512,
    "temperature": 0.7,
    "top_p": 1.00,
    "stream": stream
    }

    response = requests.post(invoke_url, headers=headers, json=payload)
    return response.json()["choices"][0]["message"]["content"]

In [ ]:
image_files = os.listdir(dir)
print(image_files)

['Islamabad_FMI_2025-01-24.png', 'Baghdad_2025-01-02.png', 'Paris_NDVI_2025-02-01.png', 'Linfen_2024-12-30.png', 'Suqian_2024-12-31.png', 'Lagos_2025-01-24.png', 'Fimiston_2025-01-09.png', 'Damascus_2025-01-14.png', 'Varanasi_2025-01-29.png', 'Xingtai_2025-01-06.png', 'Canberra_2025-01-13.png', 'Tianjiaan_2024-12-31.png', 'Paris_2024-11-28.png', 'LosAngeles_2025-01-12.png', 'London_2025-01-30.png', 'Islamabad_2025-01-24.png']


In [ ]:
#-----------Main code that generates captions for all the images in the folder across the collection of prompts---------------
print(SYSTEM_PROMPT)
for image_file in image_files:
    image_file_path = os.path.join(dir, image_file)
    for question in questions:
      prompt = create_strict_satellite_prompt(image_file, question)
      print("filename: \n", image_file)
      response = generateCaptions_llama4(image_file_path, prompt)
      print("----------RESPONSE STARTS--------------")
      print(response)
      print("----------RESPONSE ENDS----------------")
      print("--------------------------------------------------------------------------"*2)
      print("\n\n")



You are an expert assistant specializing in the interpretation of Sentinel-2 satellite imagery for environmental analysis.

You analyze spectral data from RGB and Near-Infrared bands and understand the environmental significance of band operations such as:
- NDVI (vegetation health)
- NDWI (water content)
- NDBI (built-up area)
- NBR  (burn ratio)
- FMI  (ferrous minerals)

Only describe observable environmental conditions based on the spectral content of the supplied image.

FILENAME RULES:
- The first word in the image filename is the location. Only reference this location — do not mention any other places.
- The second word indicates the image type: rgb, ndvi, ndwi, nbr, fmi, or others after the location name, it indicates a band-derived index.
- If no index is specified, the image is an RGB composite.

EXAMPLES:
- Islamabad_rgb_2024_01_24.png → RGB image of Islamabad from Janary 24th, 2024.
- Paris_ndvi_2025-02-01.png → NDVI image of Paris from Febuary 1st, 2025.
- Buenos Aires_nb

In [ ]:
print(SYSTEM_PROMPT)
for image_file in image_files:
    image_file_path = os.path.join(dir, image_file)
    for question in questions:
      prompt = LlamaPromptGenerator(image_file, question)
      print("filename: \n", image_file)
      response = generateCaptions_llama4(image_file_path, prompt)
      print("----------RESPONSE STARTS--------------")
      print(response)
      print("----------RESPONSE ENDS----------------")
      print("--------------------------------------------------------------------------"*2)
      print("\n\n")



You are an expert assistant specializing in the interpretation of Sentinel-2 satellite imagery for environmental analysis.

You analyze spectral data from RGB and Near-Infrared bands and understand the environmental significance of band operations such as:
- NDVI (vegetation health)
- NDWI (water content)
- NDBI (built-up area)
- NBR  (burn ratio)
- FMI  (ferrous minerals)

Only describe observable environmental conditions based on the spectral content of the supplied image.

FILENAME RULES:
- The first word in the image filename is the location. Only reference this location — do not mention any other places.
- The second word indicates the image type: rgb, ndvi, ndwi, nbr, fmi, or others after the location name, it indicates a band-derived index.
- If no index is specified, the image is an RGB composite.

EXAMPLES:
- Islamabad_rgb_2024_01_24.png → RGB image of Islamabad from Janary 24th, 2024.
- Paris_ndvi_2025-02-01.png → NDVI image of Paris from Febuary 1st, 2025.
- Buenos Aires_nb